# 02 — Training Curves

Load training metrics from TensorBoard event files and GPU monitor CSVs, then plot them together.

In [ ]:
import sys
sys.path.insert(0, '..')

import glob
import os
import pandas as pd
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

In [ ]:
# Point to a specific Hydra run directory, or use the latest one
OUTPUTS_ROOT = '../outputs'

# Find all TensorBoard event files
tb_event_files = glob.glob(f'{OUTPUTS_ROOT}/**/logs/tensorboard/events.out.*', recursive=True)
print(f'Found {len(tb_event_files)} TensorBoard run(s):')
for f in tb_event_files:
    print(' ', f)

In [ ]:
# Load loss and lr from TensorBoard
def load_tb_scalars(event_dir):
    ea = EventAccumulator(event_dir)
    ea.Reload()
    data = {}
    for tag in ea.Tags()['scalars']:
        events = ea.Scalars(tag)
        data[tag] = pd.DataFrame([(e.step, e.value) for e in events], columns=['step', 'value'])
    return data

if tb_event_files:
    run_dir = os.path.dirname(tb_event_files[-1])  # use latest
    tb_data = load_tb_scalars(run_dir)
    print('Available tags:', list(tb_data.keys()))

In [ ]:
# Plot training loss
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

if 'train/loss' in tb_data:
    df = tb_data['train/loss']
    axes[0].semilogy(df['step'], df['value'])
    axes[0].set_title('Training Loss (log scale)')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('MSE Loss')
    axes[0].grid(True, alpha=0.3)

if 'train/lr' in tb_data:
    df = tb_data['train/lr']
    axes[1].plot(df['step'], df['value'])
    axes[1].set_title('Learning Rate Schedule')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('LR')
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Load GPU metrics CSV
gpu_csvs = glob.glob(f'{OUTPUTS_ROOT}/**/logs/gpu/gpu_*.csv', recursive=True)
print(f'Found {len(gpu_csvs)} GPU log file(s)')

if gpu_csvs:
    gpu_df = pd.read_csv(gpu_csvs[-1], parse_dates=['timestamp'])
    print(gpu_df.describe())

In [ ]:
# Plot GPU utilisation and memory over time
if gpu_csvs and not gpu_df.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 7))
    t = gpu_df['timestamp']

    axes[0, 0].plot(t, gpu_df['gpu_util_pct'])
    axes[0, 0].set_title('GPU Utilisation (%)')
    axes[0, 0].set_ylim(0, 100)

    axes[0, 1].plot(t, gpu_df['mem_used_mb'])
    axes[0, 1].set_title('GPU Memory Used (MB)')

    axes[1, 0].plot(t, gpu_df['temp_c'])
    axes[1, 0].set_title('GPU Temperature (°C)')

    axes[1, 1].plot(t, gpu_df['power_w'])
    axes[1, 1].set_title('GPU Power Draw (W)')

    for ax in axes.flat:
        ax.grid(True, alpha=0.3)
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=20)

    plt.suptitle('GPU Metrics During Training')
    plt.tight_layout()
    plt.show()